***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [9. 实践部分](9_1_visualisation-inspection.ipynb)
    * 上一节： [9.16 成像参数选择](9_16_imaging_parameter_decision_tree.ipynb)
    * 下一节： [9.x 延伸阅读与后续实践方向](9_x_further_reading_and_workflow.ipynb)

***


## 9.17 数据质量、校准失败与图像伪影：QA 案例库

真实数据处理能力很大一部分来自失败模式识别。一个图像中出现条纹、环状残差、负碗或边缘噪声，并不意味着应该立刻更换 CLEAN 参数；它可能来自 RFI、坏天线、错误权重、带通解污染、相位跳变、缺短间距、主波束校正或自校准模型偏差。若不把这些因果链分清，后续科学测量就会建立在错误解释上。

本节把 QA 写成一个贯穿流程的案例库。它的重点不是给出一份机械检查清单，而是建立一种诊断顺序：先看原始数据，再看校准解，再看图像残差，最后才判断科学量是否可信。这样做的原因很简单：图像域看到的是所有误差叠加后的结果，而原始数据和校准解常常能更早、更清楚地暴露问题来源。


### 9.17.1 QA 是因果链，不是最后一步

数据处理链中每一步都会把前一步的错误带入下一步。RFI 或坏天线会污染校准解，坏校准解会制造图像伪影，错误图像模型又会在自校准中反过来污染增益，最终源表或通量测量便会带有系统偏差。因此，QA 应该贯穿全流程，而不是在最终图像完成后才做一次“好看不好看”的判断。

这种因果链尤其适合教学案例。每个失败模式都应同时给出四类线索：原始数据线索、校准线索、图像线索和处理动作。若只给图像现象，很容易把不同问题混为一谈；若只给软件命令，又无法训练判断能力。


![QA 因果链](figures/practical_qa_causal_chain.png)

**图 9.17.1** QA 是贯穿原始数据、校准、成像、自校准和科学产品的因果链。图像伪影出现后，应沿链路向前追踪，而不是只在图像域继续调参数。


### 9.17.2 原始数据线索：RFI、坏天线与阴影遮挡

原始数据检查最常见的图包括振幅随时间和频率的变化、相位随时间和频率的变化、按天线/基线/通道/扫描统计的 flag fraction。窄频 RFI 往往表现为固定通道范围内振幅和相位异常；坏天线常表现为所有包含该天线的基线同时异常；阴影遮挡或低仰角问题则可能表现为一段时间内振幅系统性降低。它们的共同特点是：异常在某个物理轴上具有结构，而不是完全随机出现。

若这些异常进入校准求解，求解器会尝试用天线增益或带通响应解释它们。这样得到的校准表可能看似收敛，却已经把坏数据吸收进了解。后续 applycal 会把这种错误扩散到目标场和所有基线。因此，原始数据 QA 的目标不是追求 flag 越多越好，而是把不能由物理模型解释的样本排除在求解之外，同时保留足够的 uv 覆盖和校准约束。


![原始数据 QA：振幅、相位与 flag mask](figures/practical_qa_raw_data_diagnostics.png)

**图 9.17.2** 原始数据诊断图。窄频 RFI、坏天线时段和低振幅异常在时间-频率平面上具有不同形态。结构化异常应先在数据层面处理，不能指望后续成像自动消除。


### 9.17.3 校准线索：不平滑带通、相位跳变与振幅漂移

校准解本身是重要的物理诊断。带通振幅通常应随频率平滑变化；若在少数通道出现尖峰，常说明 RFI、低信噪比通道或错误 flag 污染了带通解。时间增益相位应在合理解间隔内连续变化；若出现突然跳变，可能来自坏扫描、参考天线问题、相位校准源太弱或大气/电离层快速变化。振幅解若长期漂移，则可能影响通量标尺，尤其会污染后续幅相自校准。

校准 QA 的关键问题是“这个解是否可转移”。校准源上能拟合数据，并不等于目标场上的误差也被正确描述。带通解、时间增益、通量标尺和极化校准都有各自的物理假设；当解的形态不再符合这些假设时，应该回到数据选择、解间隔、参考天线和校准源模型，而不是直接把解应用到目标场。


![校准失败模式：带通、相位与振幅](figures/practical_qa_calibration_pathologies.png)

**图 9.17.3** 校准表中的典型异常。带通尖峰、相位跳变和振幅漂移都可能在图像域表现为伪影，但它们的处理动作不同，因此应尽量在校准表层面识别。


### 9.17.4 图像线索：条纹、环状残差、负碗与边缘噪声

图像伪影常常是最直观的报警信号，但它们不是唯一证据。条纹状结构可能来自 PSF 旁瓣、缺失或异常 flag、残余相位误差，也可能来自未清理的离轴亮源。亮源周围的环状或放射状残差常与相位校准、自校准模型、CLEAN mask 或方向相关误差有关。扩展源周围的 negative bowl 常指向短间距缺失或大尺度结构未被采样。主波束校正后的视场边缘噪声升高则是仪器响应和噪声放大的自然结果，不应简单解释为边缘存在大量弱源。

同一种图像形态可能有多个来源，因此图像 QA 应与原始数据和校准 QA 互相印证。若条纹方向与 PSF 旁瓣一致，应检查 uv 覆盖、权重和 deconvolution；若伪影随时间或天线相关，应检查校准；若只在主波束边缘出现，应检查 primary beam correction 和分析区域边界。


![常见图像伪影图谱](figures/practical_qa_image_artifact_atlas.png)

**图 9.17.4** 常见图像伪影图谱。条纹、亮源环状残差、negative bowl 和主波束边缘噪声分别指向不同的诊断路径。图像现象应作为线索，而不是作为最终解释。


### 9.17.5 从失败模式到处理动作

QA 的最终目标是决定下一步该做什么。不同失败模式对应不同动作：RFI 需要在求解前 flag 或降低权重；坏天线需要检查相关基线和参考天线；错误权重需要重新统计权重或至少重新估计图像 RMS；CLEAN mask 问题需要比较不同 mask 下的残差和通量稳定性；主波束边缘问题需要限制分析区域或使用更合适的方向相关模型。若把所有问题都用“继续 CLEAN”处理，通常只会把错误从残差转移到模型中。

下面的矩阵给出一个最小故障诊断表。它不追求覆盖所有情况，而是提供一种组织方式：每个 failure mode 同时对应 raw-data clue、calibration clue、image clue 和 action。实际项目中可以把这张表扩展成处理日志的一部分。


![QA 故障模式矩阵](figures/practical_qa_failure_mode_matrix.png)

**图 9.17.5** 最小故障模式矩阵。QA 表的作用是把现象、证据和处理动作联系起来，避免在图像域对所有问题使用同一种修补方式。


### 9.17.6 与真实工作流的对应

在 CASA 工作流中，原始数据 QA 常对应 `listobs`、`plotants`、`plotms`、`flagdata` 和 scan/antenna/channel 统计；校准 QA 对应 `plotcal`、校准表比较、参考天线检查和 apply 前后的校准源残差；图像 QA 对应 dirty/restored/residual 图、PSF、负源统计、局部 RMS 图和源表验证。WSClean、DP3、CARTA、PyBDSF、SoFiA 等工具名称不同，但同样需要保留可追踪的诊断图和处理日志。

实际教学中，最好保留失败案例，而不只保留成功案例。失败案例能训练三种能力：从图像现象回溯到数据和校准原因；在多个可能原因之间做证据比较；在修改参数后判断问题是否真的被解决，而不是被转移到另一个数据产品中。


### 9.17.7 本节结论

QA 不是处理流程的装饰，也不是发表前的最后检查。它是从原始数据到科学产品的因果追踪。可靠的科学量需要同时满足四个条件：原始数据异常得到合理处理，校准解具有物理可解释性，图像残差接近可建模噪声，最终测量对合理参数变化保持稳定。

本节的案例库还只是第一版。后续可继续加入真实望远镜数据中的 RFI、低频电离层、宽场方向相关误差、高频大气相位、短间距缺失和偏振泄漏案例，使第 9 章逐渐形成一套可用于训练的失败模式图谱。

***
